# Sprint 2 - Reranking, Hybrid Search, Query Rewriting, and HyDE

This notebook follows LS5-LS8. You will start from a weak retrieval baseline, add reranking, combine keyword and semantic signals, tune the hybrid blend, then use HyDE query rewriting to improve recall.


## 1. Install the helper core from GitHub

Run this first in Colab. It uses `%pip` to install the current helper core directly from the GitHub `main` branch before any helper imports.


In [ ]:
#@title Install helper core from GitHub
%pip install -q --upgrade "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"

print("Installed helper core from GitHub main.")


## 2. Add your OpenRouter key and imports

Embeddings, reranking, and HyDE all route through enabled OpenRouter models. Store `OPENROUTER_API_KEY` in Colab Secrets when possible; the cell below falls back to a hidden prompt.


In [ ]:
import os
from getpass import getpass


def load_openrouter_key() -> str:
    key = os.getenv("OPENROUTER_API_KEY")
    if key:
        return key

    try:
        from google.colab import userdata

        key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        key = None

    if not key:
        key = getpass("OpenRouter API key: ")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY is required for this notebook's model calls.")

    os.environ["OPENROUTER_API_KEY"] = key
    return key


_ = load_openrouter_key()
print("OpenRouter API key loaded.")


In [ ]:
import shutil
from pathlib import Path

from documents import chunk_text
from hybrid import HybridRetriever
from hyde import HyDERewriter
from keyword_search import BM25Retriever
from openrouter import OpenRouterClient, OpenRouterEmbedder
from rerank import OpenRouterReranker
from vector_store import ChromaStore

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-2")


## 3. Create the retrieval corpus and test queries

The live slides refer to a provided scaffold and test set. This notebook keeps the corpus small so the retrieval log is easy to read.


In [ ]:
COURSE_NOTES = {
    "structured_outputs": """
    Structured output turns model text into validated application data. The schema
    sits at the boundary between the flexible model response and the rest of the
    product. A valid schema protects shape, not truth.
    """,
    "rag_failure_modes": """
    Naive RAG often fails because chunks are too large, chunks are missing useful
    local context, the retriever returns generic similarity matches, or too many
    weak candidates crowd out the evidence needed for the final answer.
    """,
    "reranking": """
    Reranking is a second judgment step after first-pass retrieval. It helps when
    the right chunk is present in the candidate pool but ranked too low to reach
    the final context window.
    """,
    "hybrid_search": """
    Hybrid search combines semantic vector retrieval with lexical keyword search.
    BM25 helps exact terms like MCP, ChromaDB, HyDE, schema, and reranker stay
    visible. The blend weight is a hypothesis about which signal should matter.
    """,
    "query_rewriting": """
    Query rewriting changes the search text before retrieval. It is useful when
    the user and the source use different wording for the same intent. The rewrite
    should be the smallest useful fix that improves the evidence package.
    """,
    "tools_mcp": """
    Tool calling lets the model ask the app to run a function. MCP exposes tools
    and resources through a common server boundary. The host should inspect tools,
    call one safely, and validate the response before using it downstream.
    """,
}

TEST_QUERIES = [
    "The answer chunk exists, but generic RAG pushes it below the context cutoff. What helps?",
    "The learner asks about exact terms MCP and schema plus the meaning of safe tool calls.",
    "A student says sources use different words than their question. What retrieval fix is smallest?",
]

query = TEST_QUERIES[0]
print("Selected query:", query)


## 4. Index chunks into ChromaDB

Rebuild the local Chroma collection each time so the live demo has a clean baseline.


In [ ]:
documents = []
for source_id, text in COURSE_NOTES.items():
    documents.extend(
        chunk_text(
            text,
            source_id=source_id,
            chunk_size=450,
            overlap=80,
            metadata={"source": source_id},
        )
    )

DB_PATH = Path("/content/module_a_chroma") if Path("/content").exists() else Path(".chroma/module_a")
shutil.rmtree(DB_PATH, ignore_errors=True)

embedder = OpenRouterEmbedder(client)
store = ChromaStore(path=DB_PATH, collection_name="module_a_lessons", embedder=embedder)
indexed = store.index(documents, batch_size=8)
keyword = BM25Retriever.from_documents(store.all_documents())

print(f"Indexed {indexed} chunks into {DB_PATH}")


In [ ]:
def show_results(label, results, score_attr):
    print(f"\n{label}")
    for rank, result in enumerate(results, start=1):
        score = getattr(result, score_attr, None)
        source = result.document.metadata.get("source", "unknown")
        preview = result.document.text.replace("\n", " ")[:170]
        print(f"{rank}. {source} | {score_attr}={score}")
        print(f"   {preview}...")


def source_order(results):
    return [result.document.metadata.get("source", "unknown") for result in results]


## 5. Baseline retrieval

LS5 starts with the ranking problem: first-pass retrieval can find a useful chunk and still bury it. Inspect the candidate pool before adding another lever.


In [ ]:
baseline_candidates = store.semantic_search(query, top_k=5)
show_results("Baseline semantic retrieval", baseline_candidates, "semantic_score")
print("Baseline source order:", source_order(baseline_candidates))


## 6. LS5: Rerank the candidate pool

Reranking helps when the right evidence is already in the candidate pool. It should change the order, not invent new evidence.


In [ ]:
reranker = OpenRouterReranker(client)
reranked = reranker.rerank(query, baseline_candidates, top_n=3)
show_results("Reranked semantic candidates", reranked, "rerank_score")
print("Reranked source order:", source_order(reranked))


## 7. LS6: Add hybrid search

Hybrid search changes the first-pass candidate list by blending semantic similarity with BM25 keyword evidence.


In [ ]:
hybrid = HybridRetriever(vector_store=store, keyword_retriever=keyword, alpha=0.65)
hybrid_candidates = hybrid.search(query, top_k=5, semantic_k=8, keyword_k=8)
show_results("Hybrid retrieval, alpha=0.65", hybrid_candidates, "hybrid_score")
print("Hybrid source order:", source_order(hybrid_candidates))


## 8. LS6: Tune the blend

The blend weight is a hypothesis. Keep the query fixed, change one setting, and explain the evidence movement.


In [ ]:
for alpha in [0.25, 0.5, 0.8]:
    tuned = hybrid.search(query, top_k=3, semantic_k=8, keyword_k=8, alpha=alpha)
    print(f"alpha={alpha}: {source_order(tuned)}")


## 9. LS7: Rewrite the query with HyDE

HyDE creates a hypothetical answer document for semantic retrieval and a rewritten query for keyword retrieval. Use it when wording blocks recall.


In [ ]:
hyde = HyDERewriter(client).rewrite(
    query,
    context_hint="Module A covers structured outputs, RAG, reranking, hybrid search, query rewriting, tools, and MCP.",
)
print(hyde.model_dump_json(indent=2))

hyde_candidates = hybrid.search(
    query,
    top_k=5,
    semantic_k=8,
    keyword_k=8,
    semantic_query=hyde.hypothetical_document,
    keyword_query=hyde.rewritten_query,
)
show_results("HyDE-assisted hybrid retrieval", hyde_candidates, "hybrid_score")


## 10. LS8: Build the checkpoint defense

A Sprint 2 defense should compare controlled runs and name which retrieval lever changed the evidence package.


In [ ]:
retrieval_defense = [
    {
        "lever": "Reranking",
        "evidence": "Compare baseline_candidates to reranked. Did a useful existing chunk move up?",
    },
    {
        "lever": "Hybrid search",
        "evidence": "Compare semantic-only to hybrid_candidates. Did exact terms or meaning change the pool?",
    },
    {
        "lever": "Blend tuning",
        "evidence": "Compare alpha runs. Which signal helped this query most?",
    },
    {
        "lever": "HyDE rewriting",
        "evidence": "Compare original hybrid to HyDE-assisted hybrid. Did wording improve recall?",
    },
]

for row in retrieval_defense:
    print(f"{row['lever']}: {row['evidence']}")
